# GoSelect document processing

Run this notebook top to bottom. After editing analyzer definitions or field descriptions, set `UPDATE_ANALYZERS = True` in Settings, restart the kernel, then Run All. Analyzer setup now runs before preflight; no separate deployment call is needed. Leave the switch false when using the existing deployment.

Two paths share the same customer contract and original PDF sources:

- **Baseline, stages 1-8:** native CU sub-page classification and TEXT/SCHEDULE fields, drawing extraction, then one assembly.
- **Optional alternative, stages 9-11:** take this run's CU content and permitted original drawing images, send them together to Azure OpenAI, then compare with this run's baseline. No run paths need to be configured. It bypasses baseline field mapping and assembly, not source classification or plan exclusions.

**Saved outputs are historical, not validation of the current code.** Restart the kernel after package code changes. The 2026-09-08 source review found incomplete baseline specifications. One combined-request trial attached substantially more shared requirements to four real drives, but still misplaced minimum ampacity, omitted filter requirements and produced incomplete evidence. Neither result is approved for quotation use.

Run from the repository root in the environment containing the installed project. Credentials remain in `.env`. This notebook installs nothing. Analyzer replacement is enabled only through `UPDATE_ANALYZERS`.

Stage 3 makes read-only configuration requests and one paid CU request per uncached PDF. Stage 5 repeats paid drawing requests. After stage 8, continue to stages 9-11 in the same kernel. The alternative reuses the current CU results without calling CU again. Set `RUN_ALTERNATIVE = True` in stage 10 to authorize one Azure OpenAI request, or false to skip it. SDK retries for that request are disabled.

CU API `2026-06-01-preview` is preview. Valid JSON, field population, model evidence, confidence and returned-markdown coverage are not extraction accuracy.

## 1. Settings and inputs

Select the ordered PDFs in `GOSELECT_PDFS` (a JSON array of paths), or edit the demonstration selection in the next cell. These are inputs, not expected answers; no document-specific extraction rules are used.

**After changing analyzer definitions or field descriptions:** set `UPDATE_ANALYZERS = True` in the next cell, restart the kernel, then Run All. The notebook updates the analyzers before preflight and extraction. No separate function call or out-of-order execution is needed. Leave the switch false when no analyzer update is required, and return it to false after updating.

Setting the switch true authorizes replacement of the configured router and shared field analyzer on every run. Run All also performs paid extraction: CU analyzes uncached PDFs, drawing extraction repeats, and the alternative runs if its `RUN_ALTERNATIVE` switch is true. Analyzer changes create a new CU cache namespace.

The customer contract is the only field definition. This stage checks that every customer field is requested, not whether the service will read it correctly. `GOSELECT_CONTRACT` optionally selects another contract with the same `vfd_motor_pairs` envelope.

`REPORT_MODEL` identifies the intended CU deployment. `REPORT_DRAWING_MODEL` identifies the direct drawing deployment. `DRAWING_DPI` defaults to 300; increasing it may exceed the tile budget, which fails explicitly rather than discarding images.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display
from goselect_docproc.field_schema import extraction_schema, flat_fields, load_field_schema, native_field_paths
from goselect_docproc.render import DEFAULT_DPI

# Flag to determine whether to update analyzers during the run
UPDATE_ANALYZERS = True

ROOT = Path.cwd()
load_dotenv(ROOT / ".env", override=True)
ENDPOINT = os.environ["CONTENTUNDERSTANDING_ENDPOINT"].strip()
ANALYZER_ID = os.environ["CU_ANALYZER_ID"].strip()
MODEL = os.environ["REPORT_MODEL"].strip()
DRAWING_MODEL = os.environ["REPORT_DRAWING_MODEL"].strip()
DRAWING_DPI = int(os.getenv("DRAWING_DPI", str(DEFAULT_DPI)))
API_VERSION = "2026-06-01-preview"
if not all((ENDPOINT, ANALYZER_ID, MODEL, DRAWING_MODEL)) or DRAWING_DPI <= 0:
    raise ValueError("Endpoint, analyzer and model names must be set; drawing DPI must be positive")

PDFS = [ROOT / path for path in json.loads(os.getenv("GOSELECT_PDFS", '["sample_docs/98878_1_HoweyVFDs.pdf", "sample_docs/98878_2_HoweyOneline.pdf"]'))]
if not PDFS:
    raise ValueError("Select at least one source PDF")
sources = {f"f{index + 1}": (path.read_bytes(), path.resolve().as_uri()) for index, path in enumerate(PDFS)}
RUN_DIR = ROOT / "out" / "runs" / "notebook" / str(uuid4())
assert set(native_field_paths()) == set(load_field_schema().fields["rows"].item_definition.properties)
print("Inputs:", [path.name for path in PDFS])
print("Schema coverage:", len(flat_fields()), "customer fields requested; not an accuracy score")
print("Drawing deployment / DPI:", DRAWING_MODEL, DRAWING_DPI)
print("Run output:", RUN_DIR)

Inputs: ['98878_1_HoweyVFDs.pdf', '98878_2_HoweyOneline.pdf']
Schema coverage: 80 customer fields requested; not an accuracy score
Drawing deployment / DPI: gpt-5.6-sol 300
Run output: /workspaces/MODP/out/runs/notebook/b9742c11-55b9-46df-855d-9c6f8f1ca048


## 2. Clients

Construct the native CU client and the native OpenAI SDK drawing adapter. Use environment-provided keys or Azure Identity; no secrets are displayed.

Construction does not analyze documents. There is no text/schedule fallback or custom REST transport. The SDKs own transport retries.

In [2]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from goselect_docproc.models import AzureOpenAIModel

identity = DefaultAzureCredential()
cu_key = os.getenv("CONTENTUNDERSTANDING_API_KEY")
client = ContentUnderstandingClient(
    endpoint=ENDPOINT, credential=AzureKeyCredential(cu_key) if cu_key else identity,
    api_version=API_VERSION,
)
drawing_model = AzureOpenAIModel(
    os.getenv("AZURE_OPENAI_ENDPOINT") or ENDPOINT, DRAWING_MODEL,
    api_key=os.getenv("AZURE_OPENAI_API_KEY") or None, credential=identity,
)

## Analyzer setup (before analysis)

The next cell checks `UPDATE_ANALYZERS` from Settings. With false it does not change the deployment. With true it replaces the shared `Fields` analyzer and then the router using the current local definitions. Any update failure stops execution before analysis; preflight follows after successful setup.

The existing resource mapping supplies the CU logical model for `REPORT_MODEL`; resource-wide model defaults are never changed. Both text and schedules use the flat schema derived from the full customer contract. CU field names use double underscores instead of dots; the adapter restores the original customer paths and evidence keys.

Updating analyzers does not analyze PDFs itself. The following steps do, using a new cache namespace when analyzer definitions change. Changing only a direct OpenAI prompt does not require an analyzer update.

References: [CU SDK](https://github.com/Azure/azure-sdk-for-python/tree/main/sdk/contentunderstanding/azure-ai-contentunderstanding), [sub-page classification](https://learn.microsoft.com/azure/ai-services/content-understanding/concepts/classifier#classification-enhancements-2026-06-01-preview), [field extraction](https://learn.microsoft.com/azure/ai-services/content-understanding/document/overview#field-extraction).

In [ ]:
from goselect_docproc.producers.content_understanding import ensure_analyzer, field_analyzer, router_analyzer

if UPDATE_ANALYZERS:
    mappings = client.get_defaults().model_deployments or {}
    completion_model = next((name for name, deployment in mappings.items() if deployment == MODEL), None)
    if completion_model is None:
        raise ValueError(f"No CU completion model maps to deployment {MODEL!r}; resource defaults were not changed")
    field_id = f"{ANALYZER_ID}Fields"
    routes = {"schedule": field_id, "text": field_id}
    definitions = {
        field_id: field_analyzer(load_field_schema(), completion_model),
        ANALYZER_ID: router_analyzer(completion_model, in_page_segments=True, field_analyzer_ids=routes),
    }
    for analyzer_id, definition in definitions.items():
        ensure_analyzer(client, analyzer_id, definition)
        print("Updated:", analyzer_id)
else:
    print("Analyzer updates disabled. Preflight will check the deployed configuration.")

## 3. Preflight and analyze

The read-only preflight rejects missing routing, page-only segmentation, missing native evidence, or a deployed field schema that differs from the complete schema-derived contract. Old compact analyzers are deliberately incompatible.

After preflight, CU classifies and extracts each uncached PDF. The cache namespace includes endpoint, API/SDK versions, analyzer definitions, model mappings and extraction schema. It stores raw SDK responses; cached usage describes the original call, not new consumption. Changing drawing DPI does not invalidate CU analysis.

This is the first paid stage. Preflight validates configuration, not model suitability, field recall or service output quality.

In [ ]:
from importlib.metadata import version
from goselect_docproc.contracts import ContentType, Status
from goselect_docproc.extractors import default_extractors
from goselect_docproc.pipeline import Pipeline, PipelineConfig
from goselect_docproc.producers import ContentUnderstandingProducer
from goselect_docproc.producers.content_understanding import analyzer_configuration
from goselect_docproc.tiling import VisionLimits

analyzers = analyzer_configuration(client, ANALYZER_ID)
configuration = {
    "endpoint": ENDPOINT, "api_version": API_VERSION,
    "sdk_version": version("azure-ai-contentunderstanding"),
    "analyzers": analyzers, "model_defaults": client.get_defaults().as_dict(),
    "extraction_schema": extraction_schema(),
}
fingerprint = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()
configuration.update(drawing_dpi=DRAWING_DPI, drawing_model=DRAWING_MODEL)
print("Preflight passed: native sub-page routing and every customer field match the deployed schema")
producer = ContentUnderstandingProducer(client, analyzer_id=ANALYZER_ID, cache_dir=ROOT / "out" / "cache" / fingerprint)
pipeline = Pipeline(producer, default_extractors(drawing_model, VisionLimits.azure_openai()),
                    PipelineConfig(output_dir=RUN_DIR, drawing_dpi=DRAWING_DPI))
manifest = pipeline.segment(sources)
display(JSON({"analyzers": list(analyzers), "cache_namespace": fingerprint, "files": len(manifest.files)}))

## 4. Inspect classification and source coverage

Review every segment, its page range, native region, routed fields and service warnings. Plans are explicitly excluded; sibling text, schedules and schematic regions remain separate. OTHER content remains for review.

Preview the drawing and plan regions against the original PDF. Confirm the classification and exclusions visually, especially on mixed-content pages. Planning stops on unexplained returned markdown; this does not prove OCR captured every symbol or value.

In [ ]:
from goselect_docproc.render import render_pages
from IPython.display import Image

for segment in sorted(manifest.segments, key=manifest.sort_key):
    print(segment.segment_id, segment.content_type.value, f"pages {segment.first_page}-{segment.last_page}",
          "EXCLUDED" if segment.content_type is ContentType.PLAN else "retained", segment.section_root or "")
    print("Native region:", segment.source or "page range only")

for file_id, analysis in pipeline.analyses().items():
    print(file_id, "cached" if analysis.cache_hit else "new analysis")
    display(JSON({"coverage": manifest.coverage[file_id].model_dump(), "warnings": analysis.warnings,
                  "original_request_usage": analysis.usage, "service_infos": analysis.response.as_dict().get("infos", [])}))
    for segment_id, fields in analysis.fields_by_segment.items():
        print("Native fields:", segment_id)
        display(JSON({name: field.as_dict() for name, field in fields.items()}))

items = pipeline.plan(manifest)
for item in items:
    if item.content_type in {ContentType.DRAWING, ContentType.PLAN}:
        previews = render_pages(sources[item.file_id][0], item.first_page, item.last_page,
                                dpi=72, source=item.source, unit=item.source_unit)
        for name, preview in previews.items():
            print(item.segment_id, item.content_type.value, name)
            display(Image(data=preview, width=800))
print(len(items), "work items; coverage checks returned markdown, not OCR recall")

## 5. Extract

One generic adapter maps native CU field values into the customer row, retaining source, confidence and explicit applicability metadata. Missing or incompatible routed fields fail visibly; no reduced fallback silently fills the output with nulls.

Drawings use masked source regions and overlapping tiles with the same complete field schema. The native OpenAI response is validated, and all drawing readings require source-image review. No fuzzy tag correction, inferred rating conversion or document-specific repair runs.

Plans cost no model request. Re-running this stage repeats drawing inference charges.

In [ ]:
results = pipeline.run_all(items)
for result in sorted(results, key=lambda result: (result.file_ordinal, result.start_offset)):
    count = len(result.payload.records) if result.payload else 0
    print(result.segment_id, result.content_type.value, result.status.value, f"observations={count}")
print("Worker completion is not customer acceptance")

## 6. Validate field readings

Inspect populated readings alongside their native evidence, worker errors and exclusion reasons. Invalid customer rows, missing field evidence and model-derived drawing readings require review. Failed and empty retained segments cannot pass silently.

Confidence is uncalibrated and no field cutoff is enabled by default. Null means no value was extracted; it does not distinguish absent from unreadable. Compare the readings and missing values with the original source before accepting them.

In [ ]:
from goselect_docproc.assemble import has_value
from goselect_docproc.field_schema import get_field

for result in results:
    for error in result.errors:
        print(result.segment_id, result.status.value, error)
    if result.payload:
        for note in result.payload.notes:
            print(result.segment_id, note)
        for index, record in enumerate(result.payload.records, 1):
            display(JSON({"segment": result.segment_id, "observation": index,
                          "identified_system": record.identified_system, "motor_tag": record.motor_tag,
                          "readings": {path: get_field(record.row, path) for path in flat_fields()
                                       if has_value(get_field(record.row, path))},
                          "evidence": {path: [item.model_dump(mode="json") for item in evidence]
                                       for path, evidence in record.evidence.items()},
                          "issues": record.issues}))
print("Segments requiring attention:", sum(result.status is not Status.DONE for result in results))

## 7. Assemble once

Store identified systems separately from unassigned requirements and observations. Exact identifiers can join across sources; repeated schedule rows and different motor endpoints remain separate. Measurements retain their value, unit and qualifier together. SCHEDULE > TEXT > DRAWING selects a provisional reading, and disagreements remain visible as conflicts.

Only explicitly grounded, field-specific all-project-VFD requirements from completed text segments may apply to confirmed targets. Motor requirements and quantity never inherit this scope. Unverified scope and remaining observations stay unassigned.

The worker results retain the original readings. The assembled job is the single source for customer JSON and the review sheet. Neither export performs a second merge.

In [ ]:
job = pipeline.finish(manifest, results)
print(job.status.value, f"{job.segments_done}/{job.segments_expected} workers completed")
display(JSON({
    "systems": len(job.systems), "unassigned": len(job.unassigned),
    "failed": job.segments_failed, "review_required": job.review_required,
    "conflicts": [conflict.model_dump(mode="json") for conflict in job.conflicts],
}))
print("Original observations remain in results; assembly runs once")

## 8. Export and review

Persist configuration, raw SDK responses, usage, manifest, original worker records and assembled job. Validate customer JSON against the agreed schema before writing it. A validation failure leaves the diagnostic artifacts available.

The compact review shows populated readings with units and sources, unassigned observations, scope clauses and conflicts. All unknown customer fields remain null or empty arrays in JSON. The separate motor identifier and evidence remain in the job artifact because the customer schema has no motor-tag field.

Exported files are review artifacts, not approved quotation inputs. Source accuracy remains unverified until a reviewer checks the documents; passing offline plumbing tests does not change that.

In [ ]:
from goselect_docproc.deliver import contract_payload, review_sheet

sheet = review_sheet(job, document=", ".join(path.name for path in PDFS))
artifacts = {
    "configuration.json": configuration, "manifest.json": manifest.model_dump(mode="json"),
    "results.json": [result.model_dump(mode="json") for result in results],
    "job.json": job.model_dump(mode="json"),
}
for file_id, analysis in pipeline.analyses().items():
    artifacts[f"{file_id}-analysis.json"] = analysis.response.as_dict()
    artifacts[f"{file_id}-usage.json"] = {"cache_hit": analysis.cache_hit, "original_request": analysis.usage}
RUN_DIR.mkdir(parents=True, exist_ok=True)
for name, payload in artifacts.items():
    (RUN_DIR / name).write_text(json.dumps(payload, indent=2), encoding="utf-8")
(RUN_DIR / "review.md").write_text(sheet, encoding="utf-8")
display(Markdown(sheet))
deliverable = contract_payload(job)
assert deliverable["vfd_motor_pairs"] == [record.row for record in job.systems]
(RUN_DIR / "deliverable.json").write_text(json.dumps(deliverable, indent=2), encoding="utf-8")
print("Customer JSON schema valid:", len(deliverable["vfd_motor_pairs"]), "assembled rows")
print("Source accuracy: NOT VERIFIED. Review and approval are required before quotation use.")
print("Artifacts:", RUN_DIR)

## 9. Alternative: use the current run

After step 8, run the next cell. It takes the CU content and source references saved by **this notebook run** using `RUN_DIR`. No paths or additional environment variables need to be set. It does not call CU again.

Inspect the permitted text/image counts and masked drawing previews. TEXT, SCHEDULE and DRAWING content is retained; PLAN is excluded and OTHER is listed for review. The original PDFs must still match the saved hashes. Drawing images use the current run's DPI, native polygons and page angles, with an overview and overlapping detail tiles.

Preparation stops above 50 images, 40 detail tiles per region or 120,000 source characters rather than dropping content. The complete customer schema is unchanged. No OpenAI request is made in this step.

In [ ]:
from IPython.display import HTML, Image
from goselect_docproc.document_set import prepare_document_set, run_document_set, comparison_html, source_path

packet = None
alternative_result = None
alternative_output = None
if "RUN_DIR" not in globals() or not (RUN_DIR / "deliverable.json").is_file():
    raise ValueError("Complete step 8 in this notebook first.")

packet = prepare_document_set(RUN_DIR)
display(JSON({
    "current_run": str(RUN_DIR), "requested_deployment": DRAWING_MODEL,
    "source_regions": list(packet["sources"]), "permitted_characters": packet["permitted_source_characters"],
    "images": len(packet["images"]), "excluded": packet["excluded"],
    "max_completion_tokens": 24000, "sdk_retries": 0,
}))
for image in packet["images"]:
    if image["label"].endswith("overview"):
        print(image["source_id"], image["label"])
        display(Image(data=image["png"], width=800))
print("Current run prepared for comparison. No new CU or OpenAI request made.")

## 10. Alternative: send to OpenAI

Set **`RUN_ALTERNATIVE = True`** in the next cell and run it to make one paid request. That is the only switch. It uses the current run's CU text and permitted drawing images, the existing `DRAWING_MODEL` and Azure OpenAI connection settings, and the complete customer schema. No baseline answers are sent and CU is not called again.

The result is saved automatically under the current run's `alternatives/` folder, leaving the baseline unchanged. SDK retries are disabled. Refusal, truncation or invalid schema stops customer export while retaining diagnostics. Re-executing the cell with the switch edited to true makes another paid request; false makes no request and leaves any current alternative result available for comparison.

Continue to step 11 to compare. Model evidence remains provisional and the result stays under REVIEW; the known extraction limitations in the source-reviewed trial are not fixed by this routing change.

In [ ]:
RUN_ALTERNATIVE = True

if not RUN_ALTERNATIVE:
    print("Set RUN_ALTERNATIVE = True in this cell to send the current run to OpenAI once.")
elif packet is None or Path(packet["baseline_run"]) != RUN_DIR:
    raise ValueError("Run step 9 for the current notebook run first.")
else:
    alternative_result = None
    alternative_output = RUN_DIR / "alternatives" / str(uuid4())
    print("One paid request to", DRAWING_MODEL, "->", alternative_output)
    try:
        alternative_result = run_document_set(
            packet, alternative_output,
            endpoint=os.getenv("AZURE_OPENAI_ENDPOINT") or ENDPOINT,
            deployment=DRAWING_MODEL, api_key=os.getenv("AZURE_OPENAI_API_KEY") or None,
        )
    finally:
        RUN_ALTERNATIVE = False
    display(JSON({"status": "REVIEW", "rows": len(alternative_result["specification"]["vfd_motor_pairs"]),
                  "review_issues": alternative_result["review_issues"],
                  "unresolved_requirements": alternative_result["unresolved_requirements"]}))
    display(JSON(json.loads((alternative_output / "usage.json").read_text())))

## 11. Visual comparison and source review

Compare baseline and alternative values side by side, with changed fields and model-produced alternative evidence. Tag matching ignores case and outer whitespace only; duplicate or missing tags stay unmatched for review. Empty values are shown as blanks. A changed or populated field is not necessarily correct.

Set `SOURCE_TO_REVIEW` to any source ID printed in stage 9. When empty, the first cited drawing is selected, or the first source if no drawing is cited. The next cell shows the **original PDF page beside the exact permitted cached CU text**, followed by alternative claims citing that source. Drawing previews are rotated using the native CU page angle. The full-page preview is local review only and is never added to the model request. Drawing requests remain restricted to the stage 9 polygons. Baseline evidence remains available in the saved job and review artifacts.

This comparison is read-only: it does not change baseline artifacts, prior alternative artifacts or model values. Inspect component ownership, qualifiers, missing requirements, connections and evidence, not just JSON shape or populated-field counts. The full customer schema stays available in each deliverable, including unknown fields.

In the one source-reviewed trial on 2026-09-08, real-system rows changed from 0/4 to 4/4 for ABB/ACQ580, phases/frequency and passive filtering. The false fifth system disappeared. Minimum ampacity was still put into input-current fields, TCI and the filter warranty were absent from customer rows, and several evidence entries were incomplete. The trial took 114 seconds and 50,664 tokens. This is not an overall accuracy percentage or a controlled context-only comparison; prose model, prompt, image presentation and output budget also differed. See the README for both architectures and the bounded comparison.

In [ ]:
from base64 import b64encode
from html import escape
from io import BytesIO
from PIL import Image as PILImage
from goselect_docproc.render import render_page

SOURCE_TO_REVIEW = ""

if alternative_result is None or packet is None:
    print("Run the alternative in step 10 before comparing.")
elif Path(packet["baseline_run"]) != RUN_DIR:
    raise ValueError("The current run changed. Repeat steps 9 and 10 before comparing.")
else:
    baseline = deliverable
    display(HTML(comparison_html(baseline, alternative_result)))
    cited_sources = {name for entry in alternative_result["evidence"] for name in entry["source_ids"]}
    selected_source = SOURCE_TO_REVIEW or next(
        (name for name, source in packet["sources"].items()
         if source["category"] == "DRAWING" and name in cited_sources),
        next(iter(packet["sources"])),
    )
    if selected_source not in packet["sources"]:
        raise ValueError("Choose a source ID from stage 9")
    source = packet["sources"][selected_source]
    print("Source review:", selected_source, source["uri"], "PDF page", source["page"])
    preview = render_page(source_path(source["uri"]).read_bytes(), source["page"], dpi=150)
    if preview and source.get("rotation"):
        with PILImage.open(BytesIO(preview)) as image:
            output = BytesIO()
            image.rotate(source["rotation"], expand=True).save(output, format="PNG")
            preview = output.getvalue()
    image_html = (f'<img alt="Original PDF page" style="width:100%;height:auto" '
                  f'src="data:image/png;base64,{b64encode(preview).decode()}">' if preview else "PDF preview unavailable")
    display(HTML(
        '<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(300px,1fr));gap:16px">'
        f'<div><h4>Original PDF page (local only)</h4>{image_html}</div>'
        '<div><h4>Permitted cached CU text</h4>'
        f'<pre style="white-space:pre-wrap;overflow-wrap:anywhere">{escape(source["text"])}</pre></div></div>'
    ))
    display(JSON({"alternative_claims_for_source": [entry for entry in alternative_result["evidence"]
                                                  if selected_source in entry["source_ids"]]}))
    print("Comparison only. Values and saved artifacts are unchanged; engineering review is still required.")